# Periodic Darcy: separate face, local and reference refinement

This case follows Paredes, Valentin and Versieux (2017), section 4, DOI
[10.1090/mcom/3108](https://doi.org/10.1090/mcom/3108). On the unit square,
$K=1+100\cos^2(150x)\sin^2(150y)$, $f=\sin(x)\sin(y)$ and boundary pressure is zero.
The fixed MHM grid has 8×8 squares, local continuous Q1 fields, and 1–32 constant
normal-flux modes per face. The historical local refinement is unspecified.

The classical reference has its own refinement study. All comparisons below
are integrated full broken H1 or L2 norms, not norms of plotted sample arrays.


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm.quadrilateral import CartesianMacroMesh, solve_darcy_quadrilateral
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent


## Small independent affine patch

An affine pressure is exactly represented by Q1 locals and constant flux traces.
This portable patch runs here; the larger periodic fields are replayed from their
numerical records, without requiring a high-memory solve in the notebook.


In [ ]:
exact = lambda points: 1 + points[:, 0] - 2 * points[:, 1]
patch = solve_darcy_quadrilateral(CartesianMacroMesh(2), local_refinement=4, dirichlet=exact)
assert patch.l2_error(exact) < 1e-11
assert patch.flux_l2_error([-1, 2]) < 1e-10
print({"pressure_L2_error": patch.l2_error(exact), "residual": patch.hybrid.residual})


## Executed original-equation and replay controls

The current numerical records contain Q1/P0 fields at r128 and r256, with all
six published segmented trace spaces. Their full original volume and weak-trace
rows satisfy the declared physical-source residual criterion. The independent
full Q1/P0 assemblies agree on the same spaces. This establishes algebraic
agreement, while the r128-to-r256 full H1 increment remains about 19%.

The affine patch above is a separate invariant, not a replacement for this
heterogeneous case or a reproduction of its published curve.


In [ ]:
controls = json.loads((root / "examples/results/periodic/streamed-original-verification.json").read_text())
assert controls["accepted"]
assert {case["refinement"] for case in controls["cases"]} == {128, 256}
for case in controls["cases"]:
    rows = case["original_equation_controls"]
    assert {row["segments"] for row in rows} == {1, 2, 4, 8, 16, 32}
    for row in rows:
        original = row["final_original_equations"]
        assert original["accepted"]
        assert original["original_saddle_relative_residual"] <= controls["original_physical_source_relative_criterion"]
        replay = row["numeric_replay_digests"]
        assert replay["field_trace_coarse_kernel_digests_before"] == replay["field_trace_coarse_kernel_digests_after"]
        print(f"r={case['refinement']:4d} s={row['segments']:2d} "
              f"original relative residual={original['original_saddle_relative_residual']:.8g}")
for row in controls["local_refinement_increment"]:
    print(f"s={row['segments']:2d} r128-to-r256 "
          f"full broken H1 increment={row['relative_full_broken_h1']:.8g}")

reference = json.loads((root / "examples/results/periodic/q1-reference-verification.json").read_text())
assert reference["historical_reference_space_executed"]
assert not reference["continuum_reference_resolution_demonstrated"]
assert not reference["literature_reproduction_accepted"]
for row in reference["acquisitions"]:
    assert row["original_physical_free_row_relative_residual"] <= reference["original_criterion"]
    print(f"Conforming Q1/{row['subdivisions']} "
          f"original relative residual={row['original_physical_free_row_relative_residual']:.8g}")
for row in reference["own_spatial_refinement"]:
    print(f"Q1/{row['coarse']} to Q1/{row['fine']}: "
          f"relative full H1 increment={row['relative_h1']:.8g}, "
          f"relative L2 increment={row['relative_l2']:.8g}")


## Published curve and selected physical panels

The publication markers come from vector-PDF coordinates with an extraction
allowance. The four printed annotations occur at s1, s4, s16 and s32; s2 and
s8 retain digitization uncertainty. These allowances are distinct from
numerical reference error.

Q1/4096 executes the historical conforming reference resolution, but its
Q1/2048-to-Q1/4096 full H1 increment remains 9.77%. It is not a certified exact
or resolved baseline. The historical local subdivision count is unavailable;
matched literature reproduction and final norm-comparison panels remain pending
adequate local and reference resolution.

The selected panels retain actual macro boundaries, independent one-sided
finite-element interface values and separate signed physical Darcy-flux scales.
The raw constitutive flux is not an H(div) reconstruction.


In [ ]:
for name in ("problem", "convergence", "fields"):
    display(Image(filename=str(root / f"docs/figures/periodic/{name}.png")))


## Reproduce the larger campaign

Use `examples/periodic_reference.py --degree 1 --order 4` for guarded conforming
Q1 acquisitions; its default remains Q5/order10. The archives retain executed
basis matrices, their digests and effective coefficient/operator precision.
`examples/verify_periodic.py` owns phased MHM acquisitions,
`examples/compare_periodic.py` computes exact common-partition polynomial norms,
and `examples/plot_periodic.py` renders figures using the validated readers.

The periodic case page gives commands, quadrature and measured memory use.
Large fields remain under `build/results`; compact controls carry their actual
source generations and basis/replay checks. New acquisitions do not retag
older coefficients. Reference and local increments are sensitivities rather
than certified bounds on the unknown continuum error.
